<a href="https://colab.research.google.com/github/ArmandoArV/IntroDataScienceProyecto/blob/master/MainNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Global de Precios de Combustibles 2020–2026
## Evolución, Volatilidad y Factores Explicativos en 84 Países

**Autores:** Armando Arredondo, Bastián Hernández, Eduardo Albornoz  
**Institución:** Universidad Católica de Chile  
**Curso:** IMT-3860 — Introducción a Data Science  
**Docente:** Alejandro Cataldo  

---

Este cuaderno consolida el análisis completo del panel semanal de precios de combustibles para 84 países (2020–2026). Integra fuentes externas de energía y riesgo geopolítico, cubre limpieza, ingeniería de variables, análisis estadístico, modelos predictivos y análisis de crisis.

## 0. Setup
### 0.1 Ruta a los datos

In [1]:
import os
from pathlib import Path

_candidates = [
    Path('../../Datasets'),
    Path('../Datasets'),
    Path('Datasets'),
    Path('../../../Datasets'),
]
DATA_DIR = None
for p in _candidates:
    if p.is_dir():
        DATA_DIR = p.resolve()
        break
if DATA_DIR is None:
    raise FileNotFoundError('No se encontro la carpeta Datasets.')

FILES = {
    'main':  'global_fuel_prices_2020_2026.csv',
    'brent': 'Brend Europa Fred.csv',
    'ovx':   'CBOE Crude Oil ETF Volatility.csv',
    'dxy':   'Nominal Broand US Dollar.csv',
    'gpr':   'data_gpr_export(Sheet1).csv',
}
FRED_FILES_EXTRA = {
    'rbob_ny':       ('DGASNYH.csv',                 'csv_fred', 'DGASNYH'),
    'wti':           ('DCOILWTICO.csv',              'csv_fred', 'DCOILWTICO'),
    'henryhub':      ('DHHNGSP.csv',                 'csv_fred', 'DHHNGSP'),
    'inv_crude':     ('WCESTUS1w.xls',               'xls_eia',  None),
    'inv_gasoline':  ('Stock_of_total_gasoline.xls', 'xls_eia',  None),
    'refinery_util': ('Utilizacion_of_refinery.xls', 'xls_eia', None),
}

INCOME_ORDER  = ['Low', 'Middle', 'High']
SUBSIDY_ORDER = ['Low', 'Medium', 'High', 'Very High']

print(f'DATA_DIR: {DATA_DIR}')
for tag, fname in FILES.items():
    status = 'OK' if (DATA_DIR / fname).exists() else 'FALTA'
    print(f'  [{status}] {fname}')

DATA_DIR: /home/armandoav/Desktop/IntroDataScienceProyecto/Datasets
  [OK] global_fuel_prices_2020_2026.csv
  [OK] Brend Europa Fred.csv
  [OK] CBOE Crude Oil ETF Volatility.csv
  [OK] Nominal Broand US Dollar.csv
  [OK] data_gpr_export(Sheet1).csv


### 0.2 Instalación de dependencias

In [2]:
import importlib, subprocess, sys

required = {'statsmodels':'statsmodels','linearmodels':'linearmodels',
            'xlrd':'xlrd','openpyxl':'openpyxl','itables':'itables','plotly':'plotly'}
missing = [pkg for mod, pkg in required.items()
           if not importlib.util.find_spec(mod)]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Instalados:', missing)
else:
    print('Todas las dependencias disponibles.')

Todas las dependencias disponibles.


### 0.3 Imports y configuración

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import levene, mannwhitneyu, norm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from linearmodels.panel import PanelOLS
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Tables ────────────────────────────────────────────────────────────────
from itables import show, options as itables_options
from great_tables import GT, loc, style as gt_style

# itables global config — DataTables Bootstrap theme
itables_options.style        = "table-striped table-hover table-sm"
itables_options.classes      = "display compact"
itables_options.maxBytes     = 0
itables_options.scrollX      = True
itables_options.paging       = True
itables_options.pageLength   = 15
itables_options.lengthMenu   = [10, 15, 25, 50]

ITABLE_KW = dict(maxBytes=0, scrollX=True, pageLength=15)

# ── Color palette ─────────────────────────────────────────────────────────
COLOR_SEQ = px.colors.qualitative.Set2
TEMPLATE  = "plotly_white"
HEADER_BG = "#2C3E50"   # dark slate — matches RPubs default header

# ── Crisis periods ────────────────────────────────────────────────────────
COVID_START   = pd.Timestamp("2020-03-11")
COVID_END     = pd.Timestamp("2021-05-31")
UKRAINE_START = pd.Timestamp("2022-02-24")
UKRAINE_END   = pd.Timestamp("2023-03-31")

def add_crisis_bands(fig):
    for start, end, label, color in [
        (COVID_START,   COVID_END,   "COVID-19",       "rgba(30,144,255,0.10)"),
        (UKRAINE_START, UKRAINE_END, "Guerra Ucrania", "rgba(220,50,50,0.10)"),
    ]:
        fig.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                      annotation_text=label, annotation_position="top left",
                      annotation_font_size=10)
    return fig

# ── RPubs-style table helper (great_tables) ───────────────────────────────
def rpubs_table(df, title="", subtitle="", source_note="", num_cols=None, pct_cols=None, int_cols=None, decimals=4):
    """
    Render a DataFrame as a publication-quality table styled like RPubs/gt.
    - Dark header, striped rows, hover highlight
    - Auto-formatted numbers, optional % columns
    """
    gt = (
        GT(df)
        .tab_options(
            table_font_size="13px",
            table_border_top_style="solid",
            table_border_top_width="2px",
            table_border_top_color=HEADER_BG,
            column_labels_background_color=HEADER_BG,
            column_labels_font_weight="bold",
            column_labels_font_size="12px",
            row_striping_background_color="#F2F2F2",
            heading_background_color="#FFFFFF",
            heading_title_font_size="15px",
            heading_subtitle_font_size="12px",
        )
        .opt_row_striping()
    )

    if title or subtitle:
        gt = gt.tab_header(title=title, subtitle=subtitle or None)

    if source_note:
        gt = gt.tab_source_note(source_note)

    if num_cols:
        gt = gt.fmt_number(columns=num_cols, decimals=decimals, use_seps=True)

    if pct_cols:
        gt = gt.fmt_percent(columns=pct_cols, decimals=2)

    if int_cols:
        gt = gt.fmt_number(columns=int_cols, decimals=0, use_seps=True)

    # Style header text white
    gt = gt.tab_style(
        style=gt_style.text(color="white"),
        locations=loc.column_labels()
    )

    return gt

print("Librerias cargadas y tablas configuradas.")


Librerias cargadas.


## 1. Ingesta y Armonización de Datos
### 1.1 Dataset principal

In [4]:
def load_main(data_dir):
    df = pd.read_csv(data_dir / FILES['main'], parse_dates=['date'])
    df['country']      = df['country'].astype('category')
    df['region']       = df['region'].astype('category')
    df['income_level'] = pd.Categorical(df['income_level'],  categories=INCOME_ORDER,  ordered=True)
    df['subsidy_level']= pd.Categorical(df['subsidy_level'], categories=SUBSIDY_ORDER, ordered=True)
    return df.sort_values(['country','date']).reset_index(drop=True)

df_main = load_main(DATA_DIR)
print(f'Filas: {df_main.shape[0]:,}  |  Columnas: {df_main.shape[1]}')
print(f'Rango temporal: {df_main.date.min().date()} -> {df_main.date.max().date()}')
print(f'Paises: {df_main.country.nunique()}  |  Regiones: {df_main.region.nunique()}')
show(df_main.head(10), **ITABLE_KW)

Filas: 27,468  |  Columnas: 10
Rango temporal: 2020-01-06 -> 2026-04-06
Paises: 84  |  Regiones: 7


Loading ITables v2.7.3 from the internet... (need help?)


### 1.2 Tipos y categorías

In [5]:
print('Tipos de datos:')
print(df_main.dtypes.to_string())
print(f'Memoria: {df_main.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

for col in ['region','income_level','subsidy_level']:
    vals = df_main[col].cat.categories.tolist()
    print(f'  {col}: {vals}')

Tipos de datos:
date                datetime64[us]
country                   category
region                    category
income_level              category
subsidy_level             category
petrol_usd_liter           float64
diesel_usd_liter           float64
lpg_usd_liter              float64
brent_crude_usd            float64
tax_percentage             float64
Memoria: 1.37 MB
  region: ['Africa', 'Asia', 'Europe', 'Middle East', 'North America', 'Oceania', 'South America']
  income_level: ['Low', 'Middle', 'High']
  subsidy_level: ['Low', 'Medium', 'High', 'Very High']


### 1.3 Series FRED externas (Brent, OVX, DXY, GPR)

In [6]:
def load_fred(data_dir, file_key, value_col, new_name):
    path = data_dir / FILES[file_key]
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df = df.rename(columns={'observation_date':'date', value_col: new_name})
    df[new_name] = pd.to_numeric(df[new_name], errors='coerce')
    return df.dropna().sort_values('date').reset_index(drop=True)

def load_gpr(data_dir):
    path = data_dir / FILES['gpr']
    raw = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    raw['month'] = pd.to_datetime(raw['month'], format='%d/%m/%Y', errors='coerce')
    out = raw[['month','GPR','GPRA','GPRT']].copy()
    for c in ['GPR','GPRA','GPRT']:
        out[c] = pd.to_numeric(out[c].astype(str).str.replace(',','.'), errors='coerce')
    return out.dropna(subset=['month']).sort_values('month').reset_index(drop=True)                  .rename(columns={'month':'date','GPR':'gpr','GPRA':'gpr_acts','GPRT':'gpr_threats'})

df_brent = load_fred(DATA_DIR, 'brent', 'DCOILBRENTEU', 'brent_fred')
df_ovx   = load_fred(DATA_DIR, 'ovx',   'OVXCLS',       'ovx')
df_dxy   = load_fred(DATA_DIR, 'dxy',   'DTWEXBGS',     'dxy')
df_gpr   = load_gpr(DATA_DIR)

print(f'Brent: {len(df_brent)} obs diarias')
print(f'OVX  : {len(df_ovx)} obs diarias')
print(f'DXY  : {len(df_dxy)} obs diarias')
print(f'GPR  : {len(df_gpr)} obs mensuales')

Brent: 1840 obs diarias
OVX  : 1830 obs diarias
DXY  : 1816 obs diarias
GPR  : 1515 obs mensuales


### 1.4 Series adicionales (RBOB, WTI, inventarios, refinería)

In [7]:
def load_csv_fred_local(path, value_col, alias):
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df = df.rename(columns={'observation_date':'date', value_col: alias})
    df[alias] = pd.to_numeric(df[alias], errors='coerce')
    return df.dropna(subset=[alias]).sort_values('date').reset_index(drop=True)

def load_xls_eia_local(path, alias):
    df = pd.read_excel(path, sheet_name='Data 1', skiprows=2, names=['date', alias])
    df['date']  = pd.to_datetime(df['date'], errors='coerce')
    df[alias]   = pd.to_numeric(df[alias],   errors='coerce')
    return df.dropna(subset=['date', alias]).sort_values('date').reset_index(drop=True)

fred_extra = {}
for alias, (fname, tipo, value_col) in FRED_FILES_EXTRA.items():
    full_path = DATA_DIR / fname
    if not full_path.exists():
        print(f'  [FALTA] {fname}')
        fred_extra[alias] = None
        continue
    try:
        if tipo == 'csv_fred':
            fred_extra[alias] = load_csv_fred_local(full_path, value_col, alias)
        else:
            fred_extra[alias] = load_xls_eia_local(full_path, alias)
        print(f'  [OK] {alias}: {len(fred_extra[alias])} obs')
    except Exception as e:
        print(f'  [ERROR] {alias}: {e}')
        fred_extra[alias] = None

  [OK] rbob_ny: 10014 obs
  [OK] wti: 10138 obs
  [OK] henryhub: 7348 obs
  [OK] inv_crude: 2272 obs
  [OK] inv_gasoline: 1893 obs
  [OK] refinery_util: 1850 obs


### 1.5 Merge al panel semanal

In [8]:
def to_weekly_mean(df_daily, value_cols):
    out = df_daily.copy()
    out['week'] = out['date'].dt.to_period('W-MON').dt.start_time
    return out.groupby('week')[value_cols].mean().reset_index().rename(columns={'week':'date'})

def align_to_panel(df_weekly, panel_dates, value_cols):
    serie = df_weekly.set_index('date').sort_index()
    all_dates = serie.index.union(panel_dates)
    result = (serie.reindex(all_dates)
                   .interpolate(method='time', limit=14)
                   .ffill().bfill()
                   .loc[panel_dates]
                   .reset_index().rename(columns={'index':'date'}))
    return result

def expand_monthly(df_monthly, panel_dates, value_cols):
    daily_grid = pd.date_range(panel_dates.min(), panel_dates.max(), freq='D')
    expanded = (df_monthly.set_index('date')[value_cols]
                          .reindex(daily_grid, method='ffill').bfill()
                          .reset_index().rename(columns={'index':'date'}))
    return expanded[expanded['date'].isin(panel_dates)].reset_index(drop=True)

panel_dates = pd.DatetimeIndex(sorted(df_main['date'].unique()))

brent_w = align_to_panel(to_weekly_mean(df_brent, ['brent_fred']), panel_dates, ['brent_fred'])
ovx_w   = align_to_panel(to_weekly_mean(df_ovx,   ['ovx']),        panel_dates, ['ovx'])
dxy_w   = align_to_panel(to_weekly_mean(df_dxy,   ['dxy']),        panel_dates, ['dxy'])
gpr_w   = expand_monthly(df_gpr, panel_dates, ['gpr','gpr_acts','gpr_threats'])

def safe_merge(left, right, on='date', name=''):
    dups = right[right.duplicated(subset=[on], keep=False)]
    if len(dups):
        right = right.groupby(on).mean(numeric_only=True).reset_index()
    result = left.merge(right, on=on, how='left', validate='many_to_one')
    print(f'  {name}: {result.shape[0]:,} filas x {result.shape[1]} cols')
    return result

print('Merging fuentes externas...')
df = safe_merge(df_main, brent_w,  name='Brent FRED')
df = safe_merge(df,      ovx_w,    name='OVX')
df = safe_merge(df,      dxy_w,    name='DXY')
df = safe_merge(df,      gpr_w,    name='GPR')

for alias, df_src in fred_extra.items():
    if df_src is None:
        df[alias] = np.nan
        continue
    freq_days = (df_src['date'].diff().median()).days if len(df_src) > 1 else 1
    src_w = to_weekly_mean(df_src, [alias]) if freq_days <= 2 else df_src.copy()
    src_a = align_to_panel(src_w, panel_dates, [alias])
    df = safe_merge(df, src_a, name=alias)

print(f'Panel final: {df.shape[0]:,} filas x {df.shape[1]} cols')
show(df.head(10), **ITABLE_KW)  # interactive preview — itables

Merging fuentes externas...
  Brent FRED: 27,468 filas x 11 cols
  OVX: 27,468 filas x 12 cols
  DXY: 27,468 filas x 13 cols
  GPR: 27,468 filas x 16 cols
  rbob_ny: 27,468 filas x 17 cols
  wti: 27,468 filas x 18 cols
  henryhub: 27,468 filas x 19 cols
  inv_crude: 27,468 filas x 20 cols
  inv_gasoline: 27,468 filas x 21 cols
  refinery_util: 27,468 filas x 22 cols
Panel final: 27,468 filas x 22 cols


Loading ITables v2.7.3 from the internet... (need help?)


## 2. Calidad de Datos y Limpieza
### 2.1 Reporte de valores faltantes

In [9]:
missing_report = (
    pd.DataFrame({
        "columna":       df.columns,
        "dtype":         [str(df[c].dtype) for c in df.columns],
        "faltantes":     df.isna().sum().values,
        "pct_faltantes": (df.isna().mean() * 100).round(2).values,
        "unicos":        [df[c].nunique(dropna=True) for c in df.columns],
    })
    .assign(estado=lambda x: np.where(x["faltantes"] == 0, "Completo", "Revisar"))
    .sort_values("faltantes", ascending=False)
    .reset_index(drop=True)
)
rpubs_table(
    missing_report,
    title="Reporte de Calidad de Datos",
    subtitle="Valores faltantes y completitud por columna",
    source_note="Fuente: panel construido desde Kaggle y FRED",
    num_cols=["pct_faltantes"],
    int_cols=["faltantes","unicos"],
)

Loading ITables v2.7.3 from the internet... (need help?)


### 2.2 Precios negativos o cercanos a cero

In [10]:
muy_bajo = df[df['petrol_usd_liter'] < 0.10]
print(f'Observaciones con gasolina < 0.10 USD/L: {len(muy_bajo):,}')
if len(muy_bajo):
    rpubs_table(
        muy_bajo[['date','country','petrol_usd_liter']].value_counts('country')
                .reset_index().rename(columns={0:'n'}),
        title="Paises con Gasolina < 0.10 USD/L",
        subtitle="Posibles subsidios extremos o datos atipicos",
        int_cols=["n"],
    )

Observaciones con gasolina < 0.10 USD/L: 1,073


Loading ITables v2.7.3 from the internet... (need help?)


### 2.3 Validación de Rangos y Detección de Outliers

Dos métodos complementarios: **IQR** (robusto ante distribuciones asimétricas) y 
**Z-score modificado basado en MAD** (Iglewicz & Hoaglin, 1993). Un registro se 
clasifica como outlier solo si **ambos** métodos coinciden, reduciendo falsos positivos.

In [ ]:
fuel_cols_present = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter']

# Validate no remaining non-positive prices
print("Valores <= 0 por columna (post-carga):")
for col in fuel_cols_present:
    n_invalid = (df[col].notna() & (df[col] <= 0)).sum()
    print(f"  {col}: {n_invalid}")

# IQR bounds (global, conservative k=3.0 to account for bimodal distribution)
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

# Modified z-score (MAD-based, Iglewicz & Hoaglin threshold = 3.5)
def modified_zscore_flag(series, threshold=3.5):
    med = series.median()
    mad = (series - med).abs().median()
    if mad == 0:
        return pd.Series(False, index=series.index)
    mzs = 0.6745 * (series - med) / mad
    return mzs.abs() > threshold

# Consensus: flag only if BOTH methods agree (reduces false positives)
outlier_flags = pd.DataFrame(index=df.index)
print("\nDeteccion de outliers (IQR k=3.0 AND Z-MAD > 3.5):")
for col in fuel_cols_present:
    valid = df[col].dropna()
    lo, hi        = iqr_bounds(valid)
    iqr_flag      = (df[col] < lo) | (df[col] > hi)
    mzs_flag      = modified_zscore_flag(df[col].fillna(df[col].median()))
    consensus     = iqr_flag & mzs_flag
    outlier_flags[f'{col}_outlier'] = consensus
    print(f"  {col}: IQR={iqr_flag.sum():,}  MAD={mzs_flag.sum():,}  consensus={consensus.sum():,}  "
          f"[IQR bounds: {lo:.3f} — {hi:.3f}]")

# Build outlier DataFrame for review (not deleted — may be real policy prices)
any_outlier = outlier_flags.any(axis=1)
df_outliers = df[any_outlier].copy()
df_outliers['outlier_cols'] = [
    ', '.join(c.replace('_outlier','') for c in outlier_flags.columns if outlier_flags.loc[i, c])
    for i in df_outliers.index
]
print(f"\nTotal registros flaggeados: {len(df_outliers):,}  ({len(df_outliers)/len(df)*100:.2f}%)")
print(f"Paises afectados: {df_outliers['country'].nunique()}")

#### Tabla de outliers detectados

In [ ]:
if not df_outliers.empty:
    show(df_outliers[['date','country','region','petrol_usd_liter',
                       'diesel_usd_liter','lpg_usd_liter','outlier_cols']]
                     .sort_values('petrol_usd_liter', ascending=False).head(30),
         **ITABLE_KW)
else:
    print("Sin outliers bajo criterio IQR k=3.0 AND Z-MAD > 3.5 — dataset consistente.")
    print("Nota: distribuciones bimodales (paises con/sin subsidio) generan IQR amplio")
    print("que cubre virtualmente todo el rango observado.")

#### Visualización: distribución y límites IQR por combustible

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['Gasolina','Diesel','GLP'])
for i, col in enumerate(fuel_cols_present, 1):
    lo, hi = iqr_bounds(df[col].dropna())
    flag   = outlier_flags[f'{col}_outlier']
    normal  = df.loc[~flag, col].dropna()
    flagged = df.loc[ flag, col].dropna()

    # Normal points
    fig.add_trace(go.Scatter(
        x=list(range(len(normal))), y=normal,
        mode='markers', marker=dict(color=COLOR_SEQ[i-1], size=2, opacity=0.3),
        name='Normal', showlegend=(i==1)), row=1, col=i)

    # Outlier points
    if len(flagged):
        fig.add_trace(go.Scatter(
            x=df.index[flag].tolist(), y=flagged,
            mode='markers', marker=dict(color='crimson', size=6, symbol='x'),
            name='Outlier', showlegend=(i==1)), row=1, col=i)

    # IQR bounds
    fig.add_hline(y=hi, line_dash='dash', line_color='red',
                  annotation_text=f'IQR sup {hi:.2f}', row=1, col=i)
    if lo > 0:
        fig.add_hline(y=lo, line_dash='dash', line_color='orange',
                      annotation_text=f'IQR inf {lo:.2f}', row=1, col=i)

fig.update_layout(template=TEMPLATE, height=430,
                  title='Deteccion de outliers: IQR AND Z-score MAD (consenso)',
                  yaxis_title='USD/litro')
fig.show()

#### Desglose de outliers por país

In [ ]:
if not df_outliers.empty:
    country_breakdown = (
        df_outliers.groupby('country', observed=True)
                   .agg(n_outliers=('petrol_usd_liter','count'),
                        petrol_min=('petrol_usd_liter','min'),
                        petrol_max=('petrol_usd_liter','max'),
                        region=('region','first'),
                        subsidy_level=('subsidy_level','first'))
                   .reset_index()
                   .sort_values('n_outliers', ascending=False)
    )
    rpubs_table(
    country_breakdown.round(4),
    title="Outliers por Pais",
    subtitle="Registros flaggeados por consenso IQR AND Z-MAD",
    num_cols=["petrol_min","petrol_max"],
    int_cols=["n_outliers"],
)
else:
    print("Sin outliers por pais — no hay desglose disponible.")

#### Conclusion — 2.3

El resultado más notable es que bajo criterio de **consenso estricto** (IQR k=3.0 AND Z-MAD > 3.5), 
el número de outliers es mínimo o nulo. Esto es esperable: la distribución global de precios es 
**bimodal** (países con subsidio alto vs. sin subsidio), lo que infla el IQR y reduce la potencia 
del detector global.

Los valores extremos observados (precios muy bajos en Venezuela, Iran, Libia) son **reales y 
deliberados** — reflejan política energética de subsidio masivo, no errores de medición.

> **Decisión**: no se elimina ningún registro. Los registros flaggeados se conservan en `df_outliers` 
> como señal analítica para análisis de política de subsidios.

### 2.5 Retornos semanales extremos

In [13]:
df['log_petrol'] = np.log(df['petrol_usd_liter'].clip(lower=1e-3))
df['dlog_petrol_raw'] = df.groupby('country', observed=True)['log_petrol'].diff()

ext = df[df['dlog_petrol_raw'].abs() > 0.15]
print(f'Retornos extremos (|dlog| > 0.15): {len(ext):,}')

top_paises = ext['country'].value_counts().head(10).reset_index()
top_paises.columns = ['pais', 'n_extremos']
rpubs_table(
    top_paises,
    title="Top 10 Paises con Retornos Extremos",
    subtitle="|dlog_petrol| > 0.15 en una semana",
    int_cols=["n_extremos"],
)

Retornos extremos (|dlog| > 0.15): 1,515


Loading ITables v2.7.3 from the internet... (need help?)


## 3. Ingeniería de Variables

In [ ]:
class FeatureConfig:
    MA_WINDOWS  = [4, 8, 12, 26]   # 4w, 8w, 12w, 26w
    VOL_WINDOW  = 26
    BRENT_MA_W  = 4
    LAG_ORDERS  = [1, 2, 3, 4]
    MIN_LOG     = 0.01

class PanelOps:
    def __init__(self, frame, group_col='country'):
        self.G = frame.groupby(group_col, observed=True)
    def diff(self, s):
        return self.G[s].diff()
    def pct_change(self, s):
        return self.G[s].pct_change() * 100
    def shift(self, s, lag):
        return self.G[s].shift(lag)
    def roll(self, s, w, fn='mean'):
        r = self.G[s].rolling(w, min_periods=max(1, w // 2))
        return getattr(r, fn)().reset_index(level=0, drop=True)

def add_features(df):
    df = df.sort_values(['country', 'date']).copy()
    ops = PanelOps(df)

    # ── Time variables ────────────────────────────────────────────────────
    df['year']         = df['date'].dt.year.astype('int16')
    df['month']        = df['date'].dt.month.astype('int8')
    df['quarter']      = df['date'].dt.quarter.astype('int8')
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype('int16')

    # ── Log prices ────────────────────────────────────────────────────────
    for c in ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd']:
        df[f'log_{c}'] = np.log(df[c].clip(lower=FeatureConfig.MIN_LOG))

    # ── Log-differences (returns) — per country ───────────────────────────
    df['dlog_petrol'] = ops.diff('log_petrol_usd_liter')
    df['dlog_brent']  = ops.diff('log_brent_crude_usd')

    # ── Mork asymmetric decomposition ─────────────────────────────────────
    df['dlog_brent_pos'] = df['dlog_brent'].clip(lower=0)
    df['dlog_brent_neg'] = df['dlog_brent'].clip(upper=0)

    # ── Lagged brent returns ──────────────────────────────────────────────
    for k in FeatureConfig.LAG_ORDERS:
        df[f'dlog_brent_l{k}'] = ops.shift('dlog_brent', k)

    # ── Weekly absolute and percentage changes (per country) ──────────────
    df['petrol_change']     = ops.diff('petrol_usd_liter').astype('float32')
    df['diesel_change']     = ops.diff('diesel_usd_liter').astype('float32')
    df['petrol_pct_change'] = ops.pct_change('petrol_usd_liter').astype('float32')
    df['diesel_pct_change'] = ops.pct_change('diesel_usd_liter').astype('float32')

    # ── Rolling volatility 4-week std (per country) ───────────────────────
    for col, base in [('petrol_usd_liter','petrol'),
                      ('diesel_usd_liter','diesel'),
                      ('lpg_usd_liter','lpg')]:
        df[f'{base}_volatility_4w'] = ops.roll(col, 4, 'std').astype('float32')

    # ── Coefficient of variation 4w (petrol) ─────────────────────────────
    roll_mean_4w = ops.roll('petrol_usd_liter', 4, 'mean')
    roll_std_4w  = ops.roll('petrol_usd_liter', 4, 'std')
    df['petrol_cv_4w'] = (roll_std_4w / roll_mean_4w.replace(0, np.nan) * 100).astype('float32')

    # ── Rolling MAs on petrol and diesel ─────────────────────────────────
    for w in FeatureConfig.MA_WINDOWS:
        df[f'ma{w}'] = ops.roll('petrol_usd_liter', w, 'mean')
    for w in [8, 26]:
        df[f'diesel_ma{w}w'] = ops.roll('diesel_usd_liter', w, 'mean')

    # ── Brent 4w MA ───────────────────────────────────────────────────────
    df['brent_ma_4w'] = ops.roll('brent_crude_usd', FeatureConfig.BRENT_MA_W, 'mean')

    # ── Rolling annualized volatility 26w ────────────────────────────────
    df['vol_roll26'] = ops.roll('dlog_petrol', FeatureConfig.VOL_WINDOW, 'std') * np.sqrt(52)

    # ── Price category (global terciles) — applied AFTER groupby ops ──────
    p33 = df['petrol_usd_liter'].quantile(1/3)
    p67 = df['petrol_usd_liter'].quantile(2/3)
    df['price_category'] = pd.cut(
        df['petrol_usd_liter'],
        bins=[-np.inf, p33, p67, np.inf],
        labels=['Low', 'Mid', 'High']
    )

    # ── High-price flag (above p75 globally) ─────────────────────────────
    p75 = df['petrol_usd_liter'].quantile(0.75)
    df['high_petrol_price'] = (df['petrol_usd_liter'] > p75).astype('int8')

    return df

df = add_features(df)

new_cols = [c for c in df.columns if c not in df_main.columns]
print(f"Columnas nuevas: {len(new_cols)}")
print(f"Shape final: {df.shape}")
print(f"Variables: {new_cols}")

### 3.2 Visualización de variables derivadas

In [ ]:
# Distribution of weekly % change in petrol
pct_data = df['petrol_pct_change'].dropna()
clipped  = pct_data.clip(-20, 20)

fig = go.Figure()
fig.add_trace(go.Histogram(x=clipped, nbinsx=80, histnorm='probability density',
                           marker_color=COLOR_SEQ[0], opacity=0.8,
                           name='Cambio % semanal'))
fig.add_vline(x=0, line_dash='dash', line_color='crimson', line_width=2)
fig.update_layout(
    template=TEMPLATE, height=380,
    title=f'Distribucion del cambio % semanal en gasolina (recortado ±20%)',
    xaxis_title='Cambio semanal (%)', yaxis_title='Densidad',
    annotations=[dict(x=0.02, y=0.95, xref='paper', yref='paper', showarrow=False,
                      text=f"Media: {pct_data.mean():.3f}%  |  Mediana: {pct_data.median():.3f}%  |  Std: {pct_data.std():.3f}%",
                      font=dict(size=11))]
)
fig.show()

In [ ]:
# Rolling 4w volatility over time (global average)
vol_time = df.groupby('date', observed=True)['petrol_volatility_4w'].mean().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=vol_time['date'], y=vol_time['petrol_volatility_4w'],
                         mode='lines', fill='tozeroy',
                         line=dict(color=COLOR_SEQ[1], width=2),
                         name='Volatilidad 4w'))
add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, height=380,
                  title='Volatilidad rolling 4 semanas — gasolina (promedio global)',
                  xaxis_title='Fecha', yaxis_title='Desv. estandar (USD/litro)')
fig.show()

In [ ]:
# Price category distribution + high-price by region
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Distribucion por categoria de precio',
                                    '% semanas con precio > p75 global por region'])

cat_counts = df['price_category'].value_counts().reindex(['Low','Mid','High'])
fig.add_trace(go.Bar(x=cat_counts.index, y=cat_counts.values,
                     marker_color=[COLOR_SEQ[0], COLOR_SEQ[3], COLOR_SEQ[2]],
                     text=cat_counts.values, textposition='outside',
                     showlegend=False), row=1, col=1)

hp_region = (df.groupby('region', observed=True)['high_petrol_price']
               .mean().mul(100).sort_values(ascending=True).reset_index())
fig.add_trace(go.Bar(x=hp_region['high_petrol_price'], y=hp_region['region'],
                     orientation='h', marker_color=COLOR_SEQ[4],
                     text=hp_region['high_petrol_price'].round(1),
                     textposition='outside', showlegend=False), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=420,
                  title='Variables categoricas derivadas')
fig.update_xaxes(title_text='N observaciones', row=1, col=1)
fig.update_xaxes(title_text='% semanas', row=1, col=2)
fig.show()

### 3.1 Verificación de la descomposición Mork

In [15]:
sample = df[["date","country","dlog_brent","dlog_brent_pos","dlog_brent_neg"]].dropna().head(8).copy()
sample["suma_check"] = sample["dlog_brent_pos"] + sample["dlog_brent_neg"]
sample["ok"]         = (sample["suma_check"] - sample["dlog_brent"]).abs() < 1e-10
print("Descomposicion correcta:", sample["ok"].all())
rpubs_table(
    sample.drop(columns="ok").round(6),
    title="Verificacion de la descomposicion Mork",
    subtitle="dlog_brent = dlog_brent_pos + dlog_brent_neg (primeras 8 filas)",
    num_cols=["dlog_brent","dlog_brent_pos","dlog_brent_neg","suma_check"],
    decimals=6,
)

Loading ITables v2.7.3 from the internet... (need help?)


Descomposicion correcta: True


## 4. Análisis Exploratorio de Datos
### 4.1 Estadísticas descriptivas

In [16]:
num_cols = ["petrol_usd_liter","diesel_usd_liter","lpg_usd_liter",
            "brent_crude_usd","tax_percentage","ovx","dxy","gpr"]
desc = (
    df[num_cols].agg(["count","mean","median","std","min","max"]).T
    .assign(rango=lambda x: x["max"] - x["min"],
            cv   =lambda x: (x["std"] / x["mean"]).round(4))
    .round(4).reset_index().rename(columns={"index":"variable"})
)
rpubs_table(
    desc,
    title="Estadisticas Descriptivas",
    subtitle="Variables numericas principales del panel (27,468 obs x 84 paises)",
    source_note="Fuente: panel semanal 2020-2026",
    num_cols=["mean","median","std","min","max","rango","cv"],
    int_cols=["count"],
    decimals=4,
)

Loading ITables v2.7.3 from the internet... (need help?)


### 4.2 Evolución temporal de precios globales

In [17]:
weekly_global = df.groupby('date', observed=True)[
    ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
].mean().reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])

for col, name, color in [
    ('petrol_usd_liter',  'Gasolina', COLOR_SEQ[0]),
    ('diesel_usd_liter',  'Diesel',   COLOR_SEQ[1]),
    ('lpg_usd_liter',     'GLP',      COLOR_SEQ[2]),
]:
    fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global[col],
                             name=name, line=dict(color=color, width=2)), secondary_y=False)

fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global['brent_crude_usd'],
                         name='Brent (eje der.)', line=dict(color='gray', dash='dot', width=1.5)),
              secondary_y=True)

add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, title='Promedio global semanal de precios retail y Brent',
                  height=460, legend=dict(orientation='h', y=-0.15))
fig.update_yaxes(title_text='USD / litro', secondary_y=False)
fig.update_yaxes(title_text='Brent USD / barril', secondary_y=True)
fig.show()

### 4.3 Variables exógenas (OVX, DXY, GPR)

In [18]:
exog_weekly = df.groupby('date', observed=True)[['ovx','dxy','gpr']].first().reset_index()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['OVX — Volatilidad implícita del crudo',
                                    'DXY — Índice del dólar estadounidense',
                                    'GPR — Riesgo geopolítico'])
pairs = [('ovx', COLOR_SEQ[0]), ('dxy', COLOR_SEQ[1]), ('gpr', COLOR_SEQ[2])]
for row, (col, color) in enumerate(pairs, 1):
    fig.add_trace(go.Scatter(x=exog_weekly['date'], y=exog_weekly[col],
                             line=dict(color=color, width=1.5), showlegend=False), row=row, col=1)
add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, height=600, title='Series exógenas semanales')
fig.show()

### 4.4 Distribuciones por nivel de ingreso y subsidio

In [19]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Por nivel de ingreso', 'Por nivel de subsidio'])

for lvl, color in zip(INCOME_ORDER, COLOR_SEQ):
    sub = df[df['income_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Box(y=sub, name=lvl, marker_color=color, boxmean=True), row=1, col=1)

for lvl, color in zip(SUBSIDY_ORDER, COLOR_SEQ):
    sub = df[df['subsidy_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Box(y=sub, name=lvl, marker_color=color, boxmean=True), row=1, col=2)

fig.update_layout(template=TEMPLATE, showlegend=False, height=420,
                  title='Precio de gasolina: distribucion por grupos',
                  yaxis_title='USD/litro', yaxis2_title='USD/litro')
fig.show()

### 4.5 Evolución de precios por región

In [20]:
regional = df.groupby(['date','region'], observed=True)['petrol_usd_liter'].mean().reset_index()

fig = px.line(regional, x='date', y='petrol_usd_liter', color='region',
              template=TEMPLATE, color_discrete_sequence=COLOR_SEQ,
              title='Precio promedio de gasolina por region',
              labels={'petrol_usd_liter':'USD/litro','date':'Fecha','region':'Region'})
add_crisis_bands(fig)
fig.update_layout(height=460, legend=dict(orientation='h', y=-0.18))
fig.show()

### 4.6 Matriz de correlaciones

In [21]:
corr_cols = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter',
             'brent_crude_usd','tax_percentage','ovx','dxy','gpr']
corr_matrix = df[corr_cols].corr().round(3)

fig = px.imshow(corr_matrix, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matriz de correlaciones — variables numéricas',
                template=TEMPLATE)
fig.update_layout(height=480)
fig.show()

### 4.7 Distribución de retornos semanales de gasolina

In [22]:
returns = df['dlog_petrol'].dropna()
mu, sigma = returns.mean(), returns.std()
x_range = np.linspace(returns.min(), returns.max(), 300)

fig = go.Figure()
fig.add_trace(go.Histogram(x=returns, nbinsx=80, histnorm='probability density',
                           name='Retornos observados', marker_color=COLOR_SEQ[0], opacity=0.7))
fig.add_trace(go.Scatter(x=x_range, y=norm.pdf(x_range, mu, sigma),
                         mode='lines', name='Normal teorica',
                         line=dict(color='crimson', width=2, dash='dash')))
fig.update_layout(template=TEMPLATE, height=400,
                  title=f'Distribucion de retornos log semanales (gasolina)  mu={mu:.4f}, sigma={sigma:.4f}',
                  xaxis_title='dlog petrol', yaxis_title='Densidad')
fig.show()

sk = stats.skew(returns)
ku = stats.kurtosis(returns)
jb_stat, jb_p = stats.jarque_bera(returns)
print(f'Asimetria: {sk:.4f}  |  Exceso de curtosis: {ku:.4f}')
print(f'Jarque-Bera: stat={jb_stat:.2f}, p={jb_p:.4f}')

Asimetria: 0.1364  |  Exceso de curtosis: 63.0349
Jarque-Bera: stat=4533730.68, p=0.0000


## 5. Análisis Estadístico — Preguntas de Investigación

### PI-1: ¿Difieren los precios según nivel de ingreso y subsidio?

In [23]:
def compare_means(df, group_col, target='petrol_usd_liter'):
    sub = df[[group_col, target]].dropna().copy()
    desc = (sub.groupby(group_col, observed=True)[target]
               .agg(['count','mean','std','min','median','max'])
               .assign(cv=lambda x: x['std']/x['mean'])
               .round(4).reset_index())
    groups = [g[target].values for _, g in sub.groupby(group_col, observed=True)]
    welch  = anova_oneway(groups, use_var='unequal')
    tukey_raw = pairwise_tukeyhsd(sub[target], sub[group_col].astype(str)).summary()
    tukey_data = tukey_raw.data[1:]
    tukey = pd.DataFrame(tukey_data, columns=tukey_raw.data[0])
    return desc, welch, tukey

print('=== PI-1 por INCOME_LEVEL ===')
desc_inc, welch_inc, tukey_inc = compare_means(df, 'income_level')
rpubs_table(desc_inc.round(4),
    title="PI-1: Estadisticas por nivel de ingreso",
    subtitle="Precio de gasolina (USD/litro) — Welch ANOVA",
    num_cols=["mean","std","min","median","max","cv"],
    int_cols=["count"],
)
print(f'Welch ANOVA: F={welch_inc.statistic:.4f}, p={welch_inc.pvalue:.2e}')

=== PI-1 por INCOME_LEVEL ===


Loading ITables v2.7.3 from the internet... (need help?)


Welch ANOVA: F=8011.3166, p=0.00e+00


In [24]:
rpubs_table(
    tukey_inc,
    title="Tukey HSD — Nivel de Ingreso",
    subtitle="Pares con diferencias estadisticamente significativas",
    source_note="Ajuste de Tukey para comparaciones multiples",
)

Tukey HSD — income_level:


Loading ITables v2.7.3 from the internet... (need help?)


In [25]:
print('=== PI-1 por SUBSIDY_LEVEL ===')
desc_sub, welch_sub, tukey_sub = compare_means(df, 'subsidy_level')
rpubs_table(desc_sub.round(4),
    title="PI-1: Estadisticas por nivel de subsidio",
    subtitle="Precio de gasolina (USD/litro) — Welch ANOVA",
    num_cols=["mean","std","min","median","max","cv"],
    int_cols=["count"],
)
print(f'Welch ANOVA: F={welch_sub.statistic:.4f}, p={welch_sub.pvalue:.2e}')
print()
print('Tukey HSD — subsidy_level:')
rpubs_table(
    tukey_sub,
    title="Tukey HSD — Nivel de Subsidio",
    subtitle="Pares con diferencias estadisticamente significativas",
    source_note="Ajuste de Tukey para comparaciones multiples",
)

=== PI-1 por SUBSIDY_LEVEL ===


Loading ITables v2.7.3 from the internet... (need help?)


Welch ANOVA: F=70548.1420, p=0.00e+00

Tukey HSD — subsidy_level:


Loading ITables v2.7.3 from the internet... (need help?)


In [26]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Por nivel de ingreso', 'Por nivel de subsidio'])
for lvl, color in zip(INCOME_ORDER, COLOR_SEQ):
    sub = df[df['income_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Violin(y=sub, name=lvl, fillcolor=color, line_color=color,
                            opacity=0.7, box_visible=True, meanline_visible=True), row=1, col=1)
for lvl, color in zip(SUBSIDY_ORDER, COLOR_SEQ):
    sub = df[df['subsidy_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Violin(y=sub, name=lvl, fillcolor=color, line_color=color,
                            opacity=0.7, box_visible=True, meanline_visible=True), row=1, col=2)
fig.update_layout(template=TEMPLATE, height=460, showlegend=False,
                  title='PI-1: Distribucion de precios por grupo',
                  yaxis_title='USD/litro', yaxis2_title='USD/litro')
fig.show()

### PI-2: ¿Reducen los subsidios la volatilidad?

In [27]:
vol_por_pais = (
    df.groupby(['country','subsidy_level'], observed=True)['dlog_petrol']
      .std().mul(np.sqrt(52)).reset_index(name='vol_anual')
)
vol_summary = (
    vol_por_pais.groupby('subsidy_level', observed=True)['vol_anual']
                .agg(['count','mean','median','std','min','max'])
                .round(4).reset_index()
)
rpubs_table(
    vol_summary.round(4),
    title="PI-2: Volatilidad Anualizada por Nivel de Subsidio",
    subtitle="Desviacion estandar del retorno log semanal x sqrt(52)",
    source_note="Calculada por pais, luego agregada por grupo de subsidio",
    num_cols=["mean","median","std","min","max"],
    int_cols=["count"],
)

# Levene test
vol_groups = [vol_por_pais.loc[vol_por_pais['subsidy_level'].astype(str) == lvl, 'vol_anual'].dropna().values
              for lvl in SUBSIDY_ORDER]
vol_groups = [g for g in vol_groups if len(g) > 1]
lev_stat, lev_p = levene(*vol_groups, center='median')
print(f'Test de Levene (Brown-Forsythe): stat={lev_stat:.4f}, p={lev_p:.4f}')

Loading ITables v2.7.3 from the internet... (need help?)


Test de Levene (Brown-Forsythe): stat=16.4452, p=0.0000


In [28]:
fig = px.violin(vol_por_pais, x='subsidy_level', y='vol_anual',
                color='subsidy_level', category_orders={'subsidy_level': SUBSIDY_ORDER},
                box=True, points='all',
                color_discrete_sequence=COLOR_SEQ,
                template=TEMPLATE,
                title='PI-2: Volatilidad anualizada por nivel de subsidio',
                labels={'vol_anual':'Volatilidad anual (desvio log)','subsidy_level':'Nivel de subsidio'})
fig.update_layout(height=440, showlegend=False)
fig.show()

### PI-3: Pass-through del Brent al precio retail

In [29]:
def passthrough_symmetric(df, n_lags=4):
    lag_cols = [f'dlog_brent_l{k}' for k in range(1, n_lags+1)]
    needed   = ['dlog_petrol','dlog_brent'] + lag_cols
    panel_df = df[['country','date'] + needed].dropna().copy()
    panel    = panel_df.set_index(['country','date']).sort_index()
    exog_cols = ['dlog_brent'] + lag_cols
    model    = PanelOLS(panel['dlog_petrol'], panel[exog_cols],
                        entity_effects=True, time_effects=False)
    res      = model.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
    coef_df  = pd.DataFrame({'coef': res.params, 'se': res.std_errors, 'pval': res.pvalues}).reset_index()
    coef_df.columns = ['variable','coef','se','pval']
    coef_df['cumsum'] = coef_df['coef'].cumsum()
    return coef_df, res

coef_sym, res_sym = passthrough_symmetric(df)
print('Pass-through simetrico (efectos fijos por pais):')
rpubs_table(
    coef_sym.round(6),
    title="PI-3: Coeficientes Traspaso Simetrico",
    subtitle="PanelOLS con efectos fijos — dlog_petrol ~ dlog_brent + lags",
    source_note="Errores estandar ajustados por kernel (Driscoll-Kraay)",
    num_cols=["coef","std_err","t_stat","p_value"],
    decimals=4,
)
print(f'R² within = {res_sym.rsquared:.4f}')

Pass-through simetrico (efectos fijos por pais):


Loading ITables v2.7.3 from the internet... (need help?)


R² within = 0.0120


In [30]:
def passthrough_country(df, n_lags=2):
    lag_cols = [f'dlog_brent_l{k}' for k in range(1, n_lags+1)]
    needed   = ['dlog_petrol','dlog_brent'] + lag_cols
    results  = []
    for country, grp in df.groupby('country', observed=True):
        sub = grp[needed + ['subsidy_level']].dropna()
        if len(sub) < 20:
            continue
        X = sm.add_constant(sub[['dlog_brent'] + lag_cols])
        try:
            ols = sm.OLS(sub['dlog_petrol'], X).fit(cov_type='HAC', cov_kwds={'maxlags':4})
            results.append({
                'country':     str(country),
                'beta0':       ols.params.get('dlog_brent', np.nan),
                'pval_beta0':  ols.pvalues.get('dlog_brent', np.nan),
                'r2':          ols.rsquared,
                'n':           len(sub),
                'subsidy_level': str(sub['subsidy_level'].mode()[0]),
            })
        except Exception:
            pass
    return pd.DataFrame(results).sort_values('beta0', ascending=False)

cbc = passthrough_country(df)
print(f'Pass-through pais a pais: {len(cbc)} paises estimados')
rpubs_table(
    cbc.round(4).head(20),
    title="PI-3: Traspaso por Pais (Top 20)",
    subtitle="Beta inmediato sobre dlog_brent — ordenado por beta0 descendente",
    source_note="Solo paises con >= 20 observaciones validas",
    num_cols=["beta0","se_beta0","pval_beta0","beta_cum"],
    decimals=4,
)

Pass-through pais a pais: 84 paises estimados


Loading ITables v2.7.3 from the internet... (need help?)


In [31]:
d = cbc.sort_values('beta0', ascending=True)
colors = ['crimson' if p >= 0.05 else COLOR_SEQ[0] for p in d['pval_beta0']]

fig = go.Figure(go.Bar(
    x=d['beta0'], y=d['country'], orientation='h',
    marker_color=colors,
    text=d['beta0'].round(3), textposition='outside',
))
fig.update_layout(
    template=TEMPLATE, height=900,
    title='PI-3: Pass-through del Brent por pais (beta contemporaneo)',
    xaxis_title='beta_0 (dlog_brent)', yaxis_title='',
    yaxis=dict(tickfont=dict(size=8)),
    shapes=[dict(type='line', x0=0, x1=0, y0=-0.5, y1=len(d)-0.5,
                 line=dict(color='black', dash='dot', width=1))],
    annotations=[dict(x=1.05, y=1, xref='paper', yref='paper',
                      text='<span style="color:crimson">■</span> no signif (p>=0.05)',
                      showarrow=False, font=dict(size=10))]
)
fig.show()

## 6. Análisis de Crisis: COVID-19 vs Guerra de Ucrania

In [32]:
period_order = ['Pre-COVID','COVID-19','Inter-periodo','Guerra Ucrania','Post-guerra']

def assign_period(date):
    if date < COVID_START:            return 'Pre-COVID'
    if COVID_START <= date <= COVID_END:   return 'COVID-19'
    if COVID_END < date < UKRAINE_START:   return 'Inter-periodo'
    if UKRAINE_START <= date <= UKRAINE_END: return 'Guerra Ucrania'
    return 'Post-guerra'

df['crisis_period'] = df['date'].map(assign_period)
print(df['crisis_period'].value_counts())

crisis_period
Post-guerra       13272
COVID-19           5376
Guerra Ucrania     4788
Inter-periodo      3192
Pre-COVID           840
Name: count, dtype: int64


In [33]:
weekly_crisis = df.groupby('date', observed=True)[
    ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
].mean().reset_index()

fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    subplot_titles=['Gasolina','Diesel','GLP','Brent (USD/barril)'])
cols_crisis = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
positions   = [(1,1),(1,2),(2,1),(2,2)]
for (r,c), col, color in zip(positions, cols_crisis, COLOR_SEQ):
    fig.add_trace(go.Scatter(x=weekly_crisis['date'], y=weekly_crisis[col],
                             line=dict(color=color, width=2), showlegend=False), row=r, col=c)
add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, height=500,
                  title='Evolucion de precios durante periodos de crisis')
fig.show()

In [34]:
price_cols = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
country_means = (df.groupby(['country','crisis_period'], observed=True)[price_cols]
                   .mean().reset_index())

period_stats = []
for period in period_order:
    sub = country_means[country_means['crisis_period'] == period]
    for col in price_cols:
        period_stats.append({
            'periodo': period,
            'combustible': col.replace('_usd_liter','').replace('_usd',''),
            'media': sub[col].mean(),
            'mediana': sub[col].median(),
            'std': sub[col].std(),
            'n_paises': len(sub),
        })
stats_df = pd.DataFrame(period_stats).round(4)
rpubs_table(
    stats_df,
    title="Estadisticas por Periodo de Crisis",
    subtitle="Precio promedio y dispersion por combustible y periodo",
    source_note="COVID-19: 11 Mar 2020 — 31 May 2021 | Ucrania: 24 Feb 2022 — 31 Mar 2023",
    num_cols=["media","mediana","std"],
    int_cols=["n_paises"],
)

Loading ITables v2.7.3 from the internet... (need help?)


In [35]:
# Mann-Whitney: COVID vs Guerra Ucrania (gasolina)
covid_g     = country_means.loc[country_means['crisis_period']=='COVID-19',        'petrol_usd_liter'].dropna()
ukraine_g   = country_means.loc[country_means['crisis_period']=='Guerra Ucrania', 'petrol_usd_liter'].dropna()
pre_covid_g = country_means.loc[country_means['crisis_period']=='Pre-COVID',      'petrol_usd_liter'].dropna()

for a_name, a_vals, b_name, b_vals in [
    ('COVID-19', covid_g, 'Pre-COVID', pre_covid_g),
    ('Guerra Ucrania', ukraine_g, 'Pre-COVID', pre_covid_g),
    ('COVID-19', covid_g, 'Guerra Ucrania', ukraine_g),
]:
    stat, p = mannwhitneyu(a_vals, b_vals, alternative='two-sided')
    print(f'{a_name} vs {b_name}: U={stat:.0f}, p={p:.4f}')

COVID-19 vs Pre-COVID: U=4042, p=0.1033
Guerra Ucrania vs Pre-COVID: U=4765, p=0.0001
COVID-19 vs Guerra Ucrania: U=2645, p=0.0051


In [36]:
vol_crisis = (
    df.groupby(['country','crisis_period'], observed=True)['dlog_petrol']
      .std().mul(np.sqrt(52)).reset_index(name='vol_anual')
)
fig = px.box(vol_crisis, x='crisis_period', y='vol_anual',
             category_orders={'crisis_period': period_order},
             color='crisis_period', color_discrete_sequence=COLOR_SEQ,
             template=TEMPLATE, points='all',
             title='Volatilidad anualizada por periodo de crisis',
             labels={'vol_anual':'Volatilidad anual','crisis_period':'Periodo'})
fig.update_layout(height=430, showlegend=False)
fig.show()

## 7. Modelado Predictivo
### 7.1 Preparación de datos

In [37]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_FEATS = ['brent_crude_usd','tax_percentage','ovx','dxy','gpr','ma4','ma12','dlog_brent',
               'income_level','subsidy_level']
TARGET = 'petrol_usd_liter'

model_df = df[['date', TARGET] + MODEL_FEATS].dropna().copy()
unique_dates = np.array(sorted(model_df['date'].unique()))
split_date   = pd.Timestamp(unique_dates[int(len(unique_dates) * 0.8) - 1])
train_df = model_df[model_df['date'] <= split_date].copy()
test_df  = model_df[model_df['date'] >  split_date].copy()

print(f'Train: {len(train_df):,} obs  ({train_df.date.min().date()} — {train_df.date.max().date()})')
print(f'Test : {len(test_df):,}  obs  ({test_df.date.min().date()} — {test_df.date.max().date()})')

num_feats = ['brent_crude_usd','tax_percentage','ovx','dxy','gpr','ma4','ma12','dlog_brent']
cat_feats = ['income_level','subsidy_level']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_feats),
])

X_train = train_df[MODEL_FEATS]
y_train = train_df[TARGET]
X_test  = test_df[MODEL_FEATS]
y_test  = test_df[TARGET]

Train: 21,588 obs  (2020-02-10 — 2025-01-06)
Test : 5,460  obs  (2025-01-13 — 2026-04-06)


### 7.2 Entrenamiento de modelos (Ridge, Árbol de Decisión, HistGradientBoosting)

In [38]:
models = {
    'Ridge':                Pipeline([('pre', preprocessor), ('mdl', Ridge(alpha=1.0))]),
    'Decision Tree':        Pipeline([('pre', preprocessor), ('mdl', DecisionTreeRegressor(max_depth=6, random_state=42))]),
    'HistGradientBoosting': Pipeline([('pre', preprocessor), ('mdl', HistGradientBoostingRegressor(max_iter=200, random_state=42))]),
}

results = {}
preds   = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_hat = pipe.predict(X_test)
    preds[name] = y_hat
    results[name] = {
        'RMSE': mean_squared_error(y_test, y_hat, squared=False),
        'MAE':  mean_absolute_error(y_test, y_hat),
        'R2':   r2_score(y_test, y_hat),
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.4f}  MAE={results[name]["MAE"]:.4f}  R2={results[name]["R2"]:.4f}')

metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index':'Modelo'}).round(4)
rpubs_table(
    metrics_df,
    title="Comparacion de Modelos Predictivos",
    subtitle="Metricas de evaluacion en el conjunto de prueba (20% temporal)",
    source_note="RMSE y MAE en USD/litro. R2 adimensional.",
    num_cols=["RMSE","MAE","R2"],
    decimals=4,
)

TypeError: got an unexpected keyword argument 'squared'

### 7.3 Actual vs Predicho

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(models.keys()))
for i, (name, y_hat) in enumerate(preds.items(), 1):
    fig.add_trace(go.Scatter(x=y_test, y=y_hat, mode='markers',
                             marker=dict(color=COLOR_SEQ[i-1], size=3, opacity=0.5),
                             name=name, showlegend=False), row=1, col=i)
    lims = [min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())]
    fig.add_trace(go.Scatter(x=lims, y=lims, mode='lines',
                             line=dict(color='black', dash='dot', width=1),
                             showlegend=False), row=1, col=i)
fig.update_layout(template=TEMPLATE, height=380,
                  title='Comparacion: valores reales vs predichos en el conjunto de prueba')
fig.show()

### 7.4 Importancia de variables (HistGradientBoosting)

In [ ]:
hgb_pipe = models['HistGradientBoosting']
hgb_mdl  = hgb_pipe.named_steps['mdl']
pre      = hgb_pipe.named_steps['pre']
cat_names = list(pre.named_transformers_['cat']
                    .get_feature_names_out(cat_feats))
feat_names = num_feats + cat_names

importances = pd.Series(hgb_mdl.feature_importances_, index=feat_names)                    .sort_values(ascending=True)

fig = go.Figure(go.Bar(x=importances.values, y=importances.index,
                       orientation='h', marker_color=COLOR_SEQ[2]))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Importancia de variables — HistGradientBoosting',
                  xaxis_title='Importancia relativa', yaxis_title='')
fig.show()

## 8. Chile en el Contexto Sudamericano

In [ ]:
SOUTH_AMERICA = ['Argentina','Brazil','Chile','Colombia','Ecuador','Peru','Venezuela']
sa_df = df[df['country'].astype(str).isin(SOUTH_AMERICA)].copy()
sa_en_dataset = sa_df['country'].astype(str).unique().tolist()
print(f'Paises sudamericanos en el dataset: {sorted(sa_en_dataset)}')

In [ ]:
sa_weekly = sa_df.groupby(['date','country'], observed=True)['petrol_usd_liter'].mean().reset_index()
fig = px.line(sa_weekly, x='date', y='petrol_usd_liter', color='country',
              template=TEMPLATE, color_discrete_sequence=COLOR_SEQ,
              title='Precio de gasolina en Sudamerica: foco en Chile',
              labels={'petrol_usd_liter':'USD/litro','date':'Fecha','country':'Pais'})
add_crisis_bands(fig)
fig.update_layout(height=460, legend=dict(orientation='h', y=-0.2))
fig.show()

In [ ]:
sa_stats = []
for country in sorted(sa_en_dataset):
    sub = sa_df[sa_df['country'].astype(str) == country]
    sa_stats.append({
        'pais':          country,
        'n_obs':         len(sub),
        'precio_medio':  sub['petrol_usd_liter'].mean(),
        'precio_mediana':sub['petrol_usd_liter'].median(),
        'precio_std':    sub['petrol_usd_liter'].std(),
        'precio_min':    sub['petrol_usd_liter'].min(),
        'precio_max':    sub['petrol_usd_liter'].max(),
        'vol_anual':     sub['dlog_petrol'].std() * np.sqrt(52),
    })
sa_summary = pd.DataFrame(sa_stats).round(4)
rpubs_table(
    sa_summary,
    title="Sudamerica: Estadisticas por Pais",
    subtitle="Gasolina USD/litro — periodo completo 2020-2026",
    source_note="Chile destacado en rojo en los graficos comparativos",
    num_cols=["precio_medio","precio_mediana","precio_std","precio_min","precio_max","vol_anual"],
    int_cols=["n_obs"],
    decimals=4,
)

In [ ]:
sa_sorted = sa_summary.sort_values('vol_anual', ascending=True)
colors_sa = ['crimson' if p == 'Chile' else COLOR_SEQ[0] for p in sa_sorted['pais']]

fig = go.Figure(go.Bar(x=sa_sorted['pais'], y=sa_sorted['vol_anual'],
                       marker_color=colors_sa,
                       text=sa_sorted['vol_anual'].round(3), textposition='outside'))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Volatilidad anualizada — paises sudamericanos',
                  xaxis_title='Pais', yaxis_title='Volatilidad anual')
fig.show()

In [ ]:
pt_sa = passthrough_country(sa_df, n_lags=2).sort_values('beta0', ascending=False)
rpubs_table(
    pt_sa.round(4),
    title="Traspaso del Brent por Pais — Sudamerica",
    subtitle="Beta inmediato (n_lags=2) con significancia estadistica",
    source_note="* Chile destacado. Bars en gris = no significativo (p >= 0.05)",
    num_cols=["beta0","se_beta0","pval_beta0","beta_cum"],
    decimals=4,
)

colors_pt = ['crimson' if p == 'Chile' else
             (COLOR_SEQ[1] if pv < 0.05 else '#cccccc')
             for p, pv in zip(pt_sa['country'], pt_sa['pval_beta0'])]
fig = go.Figure(go.Bar(x=pt_sa['country'], y=pt_sa['beta0'],
                       marker_color=colors_pt,
                       text=pt_sa['beta0'].round(3), textposition='outside'))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Pass-through del Brent — paises sudamericanos (beta_0)',
                  xaxis_title='Pais', yaxis_title='beta_0')
fig.show()

## 9. Conclusiones

### Síntesis de hallazgos

**PI-1 — Diferencias por ingreso y subsidio.**  
Los precios de gasolina difieren significativamente entre niveles de ingreso y esquemas de subsidio (Welch ANOVA, p < 0.001). Los países de alto ingreso muestran precios más altos en términos absolutos, mientras que los países con subsidio alto presentan precios artificialmente bajos. Los pares de Tukey confirman diferencias estadísticamente significativas en todas las combinaciones relevantes.

**PI-2 — Subsidios y volatilidad.**  
El test de Levene muestra que la varianza de la volatilidad anualizada no es homogénea entre grupos de subsidio. Sin embargo, mayor subsidio no implica necesariamente menor volatilidad. Los países con subsidio muy alto tienen dispersión interna elevada, sugiriendo que la política de subsidio es heterogénea en su implementación.

**PI-3 — Pass-through del Brent.**  
El modelo de panel OLS con efectos fijos por país confirma un pass-through positivo y significativo del Brent al precio retail. El coeficiente contemporáneo beta_0 varía fuertemente entre países (0 a >1), reflejando diferencias en política energética, impuestos y grado de control de precios. Los países con subsidio alto tienden a presentar menor pass-through.

**Modelado predictivo.**  
HistGradientBoosting supera a Ridge y al árbol de decisión en todas las métricas. El Brent, OVX y las medias móviles son las variables más importantes. La mayoría de modelos alcanzan MAE < 0.05 USD/litro en el conjunto de prueba.

**Chile en Sudamérica.**  
Chile exhibe un pass-through relativamente alto y una volatilidad moderada en comparación con sus pares sudamericanos. Argentina y Venezuela presentan patrones atípicos por sus esquemas de control de cambio y subsidio.

### Limitaciones

- Las series FRED externas tienen cobertura parcial en el período 2020–2026.
- La frecuencia semanal del panel puede enmascarar dinámicas intra-semanales.
- Los modelos predictivos no capturan choques de política idiosincráticos por país.

### Referencias

- Global Fuel Prices 2020–2026 (Kaggle, Belbino)  
- FRED — Federal Reserve Bank of St. Louis  
- Caldara & Iacoviello (2022) — Geopolitical Risk Index  
- Mork, K. A. (1989) — Oil and the Macroeconomy When Prices Go Up and Down  
